In [1]:
import pandas as pd

# Main mobility data
df = pd.read_csv(
    "/green-projects/project-urban_colocation_intelligence/workspace/share/data/locomizer_filtered_output/canarywharf_r10_2025-03_04.csv"
)

# Timestamp fix
df['ts'] = pd.to_datetime(df['ts'], unit='s')

# Time features
df['hour'] = df['ts'].dt.hour
df['date'] = df['ts'].dt.date

In [3]:
# Workers
workers = pd.read_csv(
    "/green-projects/project-urban_colocation_intelligence/workspace/share/data/locomizer_filtered_output/likely_workers_ids.csv"
)

# Leisure
leisure = pd.read_csv(
    "/green-projects/project-urban_colocation_intelligence/workspace/share/data/locomizer_filtered_output/leisure_user_ids.csv"
)

worker_ids = workers['user_id']
leisure_ids = leisure['id']

# All users
all_ids = df['id'].unique()

In [4]:
# Ensure same datatype
df['id'] = df['id'].astype(str)
worker_ids = worker_ids.astype(str)
leisure_ids = leisure_ids.astype(str)

In [5]:
print("Total users in df:", df['id'].nunique())
print("Worker IDs:", len(worker_ids))
print("Leisure IDs:", len(leisure_ids))

print("Matched workers:", df[df['id'].isin(worker_ids)]['id'].nunique())
print("Matched leisure:", df[df['id'].isin(leisure_ids)]['id'].nunique())

Total users in df: 456567
Worker IDs: 9962
Leisure IDs: 45547
Matched workers: 9962
Matched leisure: 45547


In [6]:
def prepare_data(df, user_ids):
    
    data = df[df['id'].isin(user_ids)].copy()
    
    # Keep only stops
    data = data[data['type'] == 'stop']
    
    # Reduce noise
    data['hour_block'] = data['ts'].dt.floor('H')
    data = data.drop_duplicates(subset=['id', 'hour_block'])
    
    return data

In [7]:
df_all = prepare_data(df, all_ids)
df_workers = prepare_data(df, worker_ids)
df_leisure = prepare_data(df, leisure_ids)

print("All users:", df_all['id'].nunique())
print("Workers:", df_workers['id'].nunique())
print("Leisure:", df_leisure['id'].nunique())

/tmp/ipykernel_2223485/2377613349.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  data['hour_block'] = data['ts'].dt.floor('H')
/tmp/ipykernel_2223485/2377613349.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  data['hour_block'] = data['ts'].dt.floor('H')
/tmp/ipykernel_2223485/2377613349.py:9: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  data['hour_block'] = data['ts'].dt.floor('H')


All users: 157101
Workers: 9962
Leisure: 45547


In [8]:
import plotly.express as px

def plot_map(data, title):
    
    sample = data.sample(50000) if len(data) > 50000 else data
    
    fig = px.scatter_mapbox(
        sample,
        lat="lat",
        lon="lon",
        zoom=12,
        opacity=0.3,
        height=700
    )
    
    fig.update_layout(
        mapbox_style="carto-positron",
        title=title
    )
    
    fig.show()

In [12]:
import plotly.express as px
import json

def create_map(data, title, filename):
    
    sample = data.sample(50000, random_state=42) if len(data) > 50000 else data
    
    fig = px.scatter_mapbox(
        sample,
        lat="lat",
        lon="lon",
        zoom=12,
        height=650,
        opacity=0.3,
        title=title
    )

    # Load Canary Wharf boundary
    with open("/green-projects/project-urban_colocation_intelligence/workspace/share/data/boundary/canary_wharf.geojson") as f:
        cw_geo = json.load(f)

    fig.update_layout(
        mapbox=dict(
            style="carto-positron",
            layers=[
                {
                    "source": cw_geo,
                    "type": "line",
                    "color": "blue",
                    "line": {"width": 2},
                }
            ]
        ),
        margin={"r":0,"t":40,"l":0,"b":0}
    )

    # Save HTML
    fig.write_html(filename)
    
    print(f"Saved: {filename}")

In [13]:
create_map(df_all, "All Users Mobility", "all_users_map.html")
create_map(df_workers, "Likely Workers Mobility", "workers_map.html")
create_map(df_leisure, "Leisure Users Mobility", "leisure_map.html")

Saved: all_users_map.html
Saved: workers_map.html
Saved: leisure_map.html


In [14]:
def create_home_map(data, title, filename):
    
    sample = data.sample(5000, random_state=42)
    
    fig = px.scatter_mapbox(
        sample,
        lat="home_lat",
        lon="home_lon",
        color="distance_km",
        color_continuous_scale="YlOrRd",
        zoom=9,
        height=650,
        title=title
    )

    with open("/green-projects/project-urban_colocation_intelligence/workspace/share/data/boundary/canary_wharf.geojson") as f:
        cw_geo = json.load(f)

    fig.update_layout(
        mapbox=dict(
            style="carto-positron",
            layers=[
                {
                    "source": cw_geo,
                    "type": "line",
                    "color": "blue",
                    "line": {"width": 3},
                }
            ]
        )
    )

    fig.write_html(filename)

In [15]:
create_home_map(worker_home, "Worker Commute Distance", "worker_home_map.html")

NameError: name 'worker_home' is not defined